# 05 scikit-learn pipelines and feature selection

`AFXFeatureExtractor` is a scikit-learn transformer from audio (file paths or 1-D signals) to a feature matrix, so extraction settings take part in cross-validation and grid search. The selectors in `afx_audio.utils.selectors` then rank the columns against the labels. Needs `pip install "afx-audio[sklearn,plot]"`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from afx_audio.sklearn import AFXFeatureExtractor
from afx_audio.utils.selectors import correlation_selector, mutual_info_selector, pca_reducer
import demo_data

SR = 16000
CV = StratifiedKFold(n_splits=4, shuffle=True, random_state=0)

## Labelled data

Four classes with 16 clips each: from the configured dataset's metadata when there is one, otherwise synthetic tones, sweeps, noise bursts and AM/FM tones with random parameters and 5 to 30 dB SNR. File paths and signals are interchangeable inputs.

In [ ]:
labelled = demo_data.labelled_files(n_classes=4, n_per_class=16)
if labelled:
    X, y = labelled[0], np.array(labelled[1])
else:
    X, y = demo_data.synthetic_dataset(n_per_class=16, seconds=1.0, sr=SR)
classes, counts = np.unique(y, return_counts=True)
print(len(X), 'clips;', dict(zip(classes.tolist(), counts.tolist())))

## The transformer

`fit` resolves the config and the output columns; `transform` returns `(n_samples, n_features)`. `features` takes names (default parameters) or a dict of names to parameters; `config` also accepts a preset id such as `'speech@1'` (below).

In [ ]:
frame = {'frame_size': 1024, 'hop_length': 512}
FEATURES = {
    'mfcc': {**frame, 'n_mfcc': 13},
    'spectral_centroid': frame,
    'spectral_flatness': frame,
    'spectral_rolloff': frame,
    'zcr': frame,
    'rms_energy': frame,
    'hnr': frame,
}
extractor = AFXFeatureExtractor(features=FEATURES, sample_rate=SR, aggregation='summary')
F = extractor.fit_transform(X)
names = extractor.get_feature_names_out()
print('F:', F.shape, '| NaN cells:', int(np.isnan(F).sum()))
print(list(names[:4]), '...', list(names[-4:]))

## A pipeline

Scaling is fitted on the training folds (`StandardScaler`), not per clip. `SimpleImputer` first, because a feature that is undefined over a whole clip (HNR of silence or pure noise) is NaN, and most estimators reject NaN.

In [ ]:
pipe = make_pipeline(
    AFXFeatureExtractor(features=FEATURES, sample_rate=SR, aggregation='summary'),
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=2000),
)
scores = cross_val_score(pipe, X, y, cv=CV)
print(f'4-fold accuracy {scores.mean():.3f} +/- {scores.std():.3f} (chance {1 / len(classes):.2f})')

## A preset in the pipeline

`config='speech@1'` pins a frozen preset version, so the pipeline extracts the same columns in any AFX release that ships it. After `fit`, `provenance_` holds what a batch output records: config hash, preset id, and the AFX, NumPy, SciPy, scikit-learn and Python versions. Store it next to a saved model.

In [ ]:
preset_pipe = make_pipeline(
    AFXFeatureExtractor(config='speech@1', n_jobs=-1),
    SimpleImputer(strategy='median'),
    StandardScaler(),
    LogisticRegression(max_iter=2000),
)
scores = cross_val_score(preset_pipe, X, y, cv=CV)
print(f'speech@1, 4-fold accuracy {scores.mean():.3f} +/- {scores.std():.3f}')

preset_pipe.fit(X, y)
extractor_sp = preset_pipe[0]
print(extractor_sp.n_features_out_, 'columns:', list(extractor_sp.get_feature_names_out()[:3]), '...')
prov = extractor_sp.provenance_
print({k: prov[k] for k in ('preset_id', 'preset_modified', 'config_hash', 'sample_rate')})
print(prov['versions'])

`preset_modified` becomes True when a constructor argument changes a value the preset sets, here the aggregation, and the config hash changes with it.

In [ ]:
changed = AFXFeatureExtractor(config='speech@1', aggregation='mean').fit(X[:1])
for name, est in (('speech@1', extractor_sp), ("speech@1, aggregation='mean'", changed)):
    print(f"{name:30s} preset_modified={est.provenance_['preset_modified']!s:5s} "
          f"config_hash={est.provenance_['config_hash']}")

## Grid search over extraction settings

Extraction parameters are hyperparameters like any other. Here: aggregation method and MFCC analysis window. Each candidate re-extracts the features, so keep grids small or cache a matrix per setting.

In [ ]:
def with_frame(size):
    f = {'frame_size': size, 'hop_length': size // 2}
    return {name: {**params, **f} for name, params in FEATURES.items()}


grid = GridSearchCV(
    pipe,
    {
        'afxfeatureextractor__aggregation': ['mean', 'summary'],
        'afxfeatureextractor__features': [with_frame(512), with_frame(2048)],
    },
    cv=CV,
)
grid.fit(X, y)
for params, score in zip(grid.cv_results_['params'], grid.cv_results_['mean_test_score']):
    size = params['afxfeatureextractor__features']['mfcc']['frame_size']
    print(f"{params['afxfeatureextractor__aggregation']:8s} frame_size={size:5d}  accuracy {score:.3f}")

## Which columns matter

`get_feature_names_out()` keeps model inspection readable.

In [ ]:
F_imp = SimpleImputer(strategy='median').fit_transform(F)
forest = RandomForestClassifier(n_estimators=300, random_state=0).fit(F_imp, y)
top = np.argsort(forest.feature_importances_)[::-1][:12]
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(range(top.size), forest.feature_importances_[top])
ax.set_xticks(range(top.size), names[top], rotation=45, ha='right')
ax.set_ylabel('impurity importance')
ax.set_title('Random forest, top 12 columns')
fig.tight_layout()

## Selectors

On the imputed and standardized matrix with the real labels. `mutual_info_selector` ranks columns by mutual information with the labels, `correlation_selector` drops a column that correlates above a threshold with an earlier one, and `pca_reducer` projects to principal components.

In [ ]:
F_std = StandardScaler().fit_transform(F_imp)

mi_idx = mutual_info_selector(F_std, y, k=10, random_state=0)
print('top 10 by mutual information:', list(names[mi_idx]))

kept = correlation_selector(F_std, threshold=0.9)
print(f'correlation filter at 0.9 keeps {len(kept)} of {F.shape[1]} columns')
for t in (0.7, 0.8, 0.95, 0.99):
    print(f'  threshold {t}: {len(correlation_selector(F_std, threshold=t))} columns')

In [ ]:
pcs, pca = pca_reducer({'features': F_std.T}, n_components=2)
fig, ax = plt.subplots(figsize=(5.5, 4.5))
for label in classes:
    m = y == label
    ax.scatter(pcs[m, 0], pcs[m, 1], s=18, label=label)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.0%})')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.0%})')
ax.legend(fontsize=8)
fig.tight_layout()

## Selection inside cross-validation

Ranking columns on all clips and then cross-validating leaks the test folds into the selection. Fit the selector on each training fold only.

In [ ]:
def cv_accuracy(select=None):
    accs = []
    for train, test in CV.split(F, y):
        imp = SimpleImputer(strategy='median').fit(F[train])
        scaler = StandardScaler().fit(imp.transform(F[train]))
        tr, te = scaler.transform(imp.transform(F[train])), scaler.transform(imp.transform(F[test]))
        cols = list(range(F.shape[1])) if select is None else select(tr, y[train])
        clf = LogisticRegression(max_iter=2000).fit(tr[:, cols], y[train])
        accs.append(clf.score(te[:, cols], y[test]))
    return np.mean(accs), len(cols)


for label, select in [
    ('all columns', None),
    ('mutual information, k=10', lambda Xt, yt: mutual_info_selector(Xt, yt, k=10, random_state=0)),
    ('correlation filter, 0.9', lambda Xt, yt: correlation_selector(Xt, threshold=0.9)),
]:
    acc, n = cv_accuracy(select)
    print(f'{label:26s} {n:3d} columns  accuracy {acc:.3f}')